# k8b — Ứng viên khó + nhãn teacher cho câu hỏi tổng hợp (GPU T4 ×2, vLLM)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, output k8a (`mir2-synth2`), output k7 có `runs/<VARIANT>_lin07.parquet`.
Mỗi câu tổng hợp: đoạn nguồn + đoạn tốt nhất của top-`TOP` tài liệu (âm khó) + `FAR` tài liệu ngẫu nhiên hạng 50–200
(âm dễ); bỏ bản gần trùng đoạn nguồn. Teacher chấm 0/1/2 (tắt chế độ suy nghĩ), lấy xác suất từng mức từ logprob → nhãn mềm.
**v2 (mặc định `LEVEL = 'doc'`):** teacher xem tiêu đề + phần đầu bài + đoạn liên quan nhất và chấm CẢ TÀI LIỆU, tiêu chí
chặt hơn (trang hỏi-đáp về ca của người khác tối đa 1). v1: `LEVEL = 'unit'` (chấm từng đoạn).
**Nhiều tài khoản:** cùng Input, `SHARD = '0/4'` … `'3/4'` (ứng viên tính lại giống hệt ở mọi phiên).
**Chạy thử:** `LIMIT = 30` (30 câu) → xem cặp/s ở log để chia phần.
**Đầu ra:** `<VARIANT>/pairs.parquet`, `<VARIANT>/labels-<phần>.parquet` → *New Dataset* → k8c.
**vLLM trên T4:** `VLLM_SPEC` ghim bản dùng PyTorch CUDA 12.x như Kaggle (bản vLLM mới nhất kéo PyTorch CUDA
13 → lệch torchaudio của Kaggle và cần driver ≥ 580). Ô cài đặt gỡ torchaudio (không dùng) và dừng nếu torch mới không
thấy GPU. Lỗi khi nạp model → thử `VLLM_SPEC = 'vllm==0.11.0'`, hoặc `MODEL = 'Qwen/Qwen3-14B-AWQ'` (nhanh ~2 lần, chất
lượng thấp hơn).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8b-v3 (15/10/2026)'
VARIANT = 'synth2'                   # 'synth' = v1; 'synth2' = v2
LEVEL = 'doc'                        # 'doc' = chấm cả tài liệu (v2); 'unit' = chấm từng đoạn (v1)
MODEL = 'Qwen/Qwen3-32B-AWQ'
TOP, FAR = 12, 4
SHARD = '0/1'
LIMIT = 30                           # >0: chỉ N câu đầu của phần (đo tốc độ); 0 = cả phần
VLLM_SPEC = 'vllm==0.9.2'            # PyTorch 2.7 (CUDA 12.x), T4 chạy engine V0
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) Dữ liệu k8a + run gộp của k7 → ứng viên (CPU; v2 đọc thêm phần đầu bài trong kho)
for d in locate_all(f'{VARIANT}/source.parquet'):
    shutil.copytree(f'{d}/{VARIANT}', f'{K.work}/{VARIANT}', dirs_exist_ok=True)
for d in locate_all(f'qexp/{VARIANT}/queries_vi.parquet'):
    shutil.copytree(f'{d}/qexp/{VARIANT}', f'{K.work}/qexp/{VARIANT}', dirs_exist_ok=True)
RUN = locate(f'runs/{VARIANT}_lin07.parquet') / 'runs' / f'{VARIANT}_lin07.parquet'
print('run:', RUN)
VA = f' --variant {VARIANT}'
sh(f'python -m mirret.synth candidates --config {K.cfg} --run {RUN} --top {TOP} --far {FAR}{VA}', 'candidates')

In [ ]:
# (3) Teacher chấm (2 GPU, tensor parallel)
sh('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
sh(f'python -m pip install -q "{VLLM_SPEC}"', 'pip_vllm')
sh('python -m pip uninstall -y -q torchaudio')       # vLLM thay torch của Kaggle; torchaudio cũ lệch CUDA, không dùng tới
sh('python -c "import torch, vllm; print(vllm.__version__, torch.__version__, torch.version.cuda, torch.cuda.is_available())"')
try:
    sh('python -c "import sys, torch; sys.exit(0 if torch.cuda.is_available() else 3)"')
except Exception:
    raise SystemExit('torch mới không thấy GPU (driver Kaggle không hỗ trợ CUDA của bản torch này) → đổi VLLM_SPEC')
lim = f' --limit {LIMIT}' if LIMIT else ''
sh(f'python -m mirret.synth label --config {K.cfg} --model {MODEL} --tp 2 --shard {SHARD} --level {LEVEL}{lim}{VA}', 'label')
report('k8b')